# Notebook 21 — serving_fornecedor_features

**Purpose:** Build the ML feature table for H2O Driverless AI classification.  
One row per supplier (`cnpj_token`). Numeric and low-cardinality categorical features only.  
No free-text fields. No high-cardinality identifiers.

**Input:**
- `data/gold/dim_fornecedor.parquet`
- `data/gold/fato_contratos.parquet`
- `data/gold/fato_sancoes.parquet`

**Output:**
- `data/serving/serving_fornecedor_features.parquet`

**Steps:**
1. Imports and configuration
2. Load Gold tables
3. Build supplier base (dim_fornecedor snapshot — active records only)
4. Aggregate contract features
5. Aggregate sanction features
6. Assemble final table and write Parquet
7. Validation and checks

**Notes:**
- Grain: one row per cnpj_token
- All binary features default to 0 (not NULL) when no match exists
- No cnpj_normalized or razao_social exposed — ADR-005 and ADR-008
- Consumer: H2O Driverless AI AutoML

In [ ]:
## Step 1 — Imports and configuration

import duckdb
from pathlib import Path
import sys

# ── project root ──────────────────────────────────────────────────────────────
ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "utils" / "paths.py").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Project root not found")

sys.path.insert(0, str(ROOT))

from utils.paths import (
    get_project_root,
    gold_path,
    serving_path,
    to_sql_path,
    ensure_dir,
)

ROOT = get_project_root()

# ── paths ─────────────────────────────────────────────────────────────────────
DIM_FORNECEDOR_PATH  = gold_path(ROOT, "dim_fornecedor")
FATO_CONTRATOS_PATH  = gold_path(ROOT, "fato_contratos")
FATO_SANCOES_PATH    = gold_path(ROOT, "fato_sancoes")
OUTPUT_PATH          = serving_path(ROOT, "serving_fornecedor_features")

ensure_dir(ROOT / "data" / "serving")

# ── connection ────────────────────────────────────────────────────────────────
con = duckdb.connect()

print("ROOT          :", ROOT)
print("DIM_FORNECEDOR:", DIM_FORNECEDOR_PATH, "| exists:", Path(DIM_FORNECEDOR_PATH).exists())
print("FATO_CONTRATOS:", FATO_CONTRATOS_PATH, "| exists:", Path(FATO_CONTRATOS_PATH).exists())
print("FATO_SANCOES  :", FATO_SANCOES_PATH,   "| exists:", Path(FATO_SANCOES_PATH).exists())
print("OUTPUT        :", OUTPUT_PATH)

In [ ]:
## Step 2 — Load Gold tables

con.execute(f"""
    CREATE OR REPLACE VIEW dim_fornecedor AS
    SELECT * FROM read_parquet('{DIM_FORNECEDOR_PATH}')
""")

con.execute(f"""
    CREATE OR REPLACE VIEW fato_contratos AS
    SELECT * FROM read_parquet('{FATO_CONTRATOS_PATH}')
""")

con.execute(f"""
    CREATE OR REPLACE VIEW fato_sancoes AS
    SELECT * FROM read_parquet('{FATO_SANCOES_PATH}')
""")

# Sanity check — row counts
for table in ["dim_fornecedor", "fato_contratos", "fato_sancoes"]:
    n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {n:,} rows")

In [ ]:
## Step 3 — Build supplier base (active snapshot from dim_fornecedor)

con.execute("""
    CREATE OR REPLACE TABLE supplier_base AS
    SELECT
        supplier_sk,
        cnpj_normalized,
        porte,
        natureza_juridica_desc,
        uf,
        situacao_cadastral,
        valid_from,
        valid_to,
        is_current
    FROM dim_fornecedor
    WHERE is_current = true
""")

n = con.execute("SELECT COUNT(*) FROM supplier_base").fetchone()[0]
print(f"supplier_base: {n:,} rows")

# Verificar distribuição de porte — confirmar categorias presentes
print("\nPorte distribution:")
print(con.execute("""
    SELECT porte, COUNT(*) AS qtd
    FROM supplier_base
    GROUP BY porte
    ORDER BY qtd DESC
""").df().to_string())

In [ ]:
## Step 4 — Aggregate contract features (from fato_contratos)

con.execute("""
    CREATE OR REPLACE TABLE contract_features AS
    SELECT
        supplier_sk,

        -- volume
        COUNT(*)                                        AS qtd_contratos,
        COUNT(DISTINCT codigo_orgao)                    AS qtd_orgaos_distintos,

        -- valor (excluir registros com valor negativo — dados sujos da fonte)
        SUM(CASE WHEN NOT _valor_negativo THEN valor_inicial ELSE 0 END)
                                                        AS valor_total_contratos,
        AVG(CASE WHEN NOT _valor_negativo THEN valor_inicial END)
                                                        AS valor_medio_contrato,
        MAX(CASE WHEN NOT _valor_negativo THEN valor_inicial END)
                                                        AS valor_max_contrato,

        -- span de atividade em anos
        DATEDIFF('year',
            MIN(data_referencia),
            MAX(data_referencia))                       AS anos_ativo_contratos,

        -- presença por fonte
        MAX(CASE WHEN fonte = 'pncp'       THEN 1 ELSE 0 END) AS tem_contrato_pncp,
        MAX(CASE WHEN fonte = 'compras_gov' THEN 1 ELSE 0 END) AS tem_contrato_compras

    FROM fato_contratos
    GROUP BY supplier_sk
""")

n = con.execute("SELECT COUNT(*) FROM contract_features").fetchone()[0]
print(f"contract_features: {n:,} rows")

# Verificar valores nulos e ranges
print(con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE valor_total_contratos IS NULL) AS null_valor,
        MIN(qtd_contratos)        AS min_contratos,
        MAX(qtd_contratos)        AS max_contratos,
        MIN(valor_total_contratos) AS min_valor,
        MAX(valor_total_contratos) AS max_valor
    FROM contract_features
""").df().to_string())

In [ ]:
## Step 5 — Aggregate sanction features (from fato_sancoes)

con.execute("""
    CREATE OR REPLACE TABLE sanction_features AS
    SELECT
        supplier_sk,

        -- presença e volume
        1                                               AS tem_sancao,
        COUNT(*)                                        AS qtd_sancoes,

        -- por fonte
        MAX(CASE WHEN fonte = 'ceis' THEN 1 ELSE 0 END) AS tem_ceis,
        MAX(CASE WHEN fonte = 'cnep' THEN 1 ELSE 0 END) AS tem_cnep,

        -- vigência
        MAX(CASE WHEN is_ativa = true THEN 1 ELSE 0 END) AS sancao_ativa,

        -- multa (CNEP only — NULL quando não há multa registrada)
        SUM(valor_multa)                                AS valor_total_multa

    FROM fato_sancoes
    GROUP BY supplier_sk
""")

n = con.execute("SELECT COUNT(*) FROM sanction_features").fetchone()[0]
print(f"sanction_features: {n:,} rows")

print(con.execute("""
    SELECT
        SUM(tem_ceis)    AS com_ceis,
        SUM(tem_cnep)    AS com_cnep,
        SUM(sancao_ativa) AS com_sancao_ativa,
        COUNT(CASE WHEN valor_total_multa IS NOT NULL THEN 1 END) AS com_multa
    FROM sanction_features
""").df().to_string())

In [ ]:
## Step 6 — Assemble final table and write Parquet

con.execute(f"""
    COPY (
        SELECT
            -- chave
            b.supplier_sk,

            -- identidade (categórico)
            b.porte,
            b.natureza_juridica_desc,
            b.uf,
            b.situacao_cadastral,

            -- features contratuais (numérico)
            COALESCE(c.qtd_contratos,       0)          AS qtd_contratos,
            COALESCE(c.qtd_orgaos_distintos, 0)         AS qtd_orgaos_distintos,
            COALESCE(c.valor_total_contratos, 0.0)      AS valor_total_contratos,
            COALESCE(c.valor_medio_contrato,  0.0)      AS valor_medio_contrato,
            COALESCE(c.valor_max_contrato,    0.0)      AS valor_max_contrato,
            COALESCE(c.anos_ativo_contratos,  0)        AS anos_ativo_contratos,
            COALESCE(c.tem_contrato_pncp,     0)        AS tem_contrato_pncp,
            COALESCE(c.tem_contrato_compras,  0)        AS tem_contrato_compras,

            -- features de sanção (binário/numérico)
            COALESCE(s.tem_sancao,        0)            AS tem_sancao,
            COALESCE(s.qtd_sancoes,       0)            AS qtd_sancoes,
            COALESCE(s.tem_ceis,          0)            AS tem_ceis,
            COALESCE(s.tem_cnep,          0)            AS tem_cnep,
            COALESCE(s.sancao_ativa,      0)            AS sancao_ativa,
            COALESCE(s.valor_total_multa, 0.0)          AS valor_total_multa,

            -- metadata
            CURRENT_TIMESTAMP                           AS _loaded_at

        FROM supplier_base b
        LEFT JOIN contract_features c USING (supplier_sk)
        LEFT JOIN sanction_features  s USING (supplier_sk)

        -- cnpj_normalized excluído intencionalmente — ADR-005
    )
    TO '{OUTPUT_PATH}'
    (FORMAT PARQUET)
""")

print(f"Written: {OUTPUT_PATH}")
print(f"File size: {Path(OUTPUT_PATH).stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
## Step 7 — Validation and checks

result = con.execute(f"""
    SELECT * FROM read_parquet('{OUTPUT_PATH}')
""")

con.execute(f"CREATE OR REPLACE VIEW features AS SELECT * FROM read_parquet('{OUTPUT_PATH}')")

checks = []

# Check 1 — row count igual ao universo de fornecedores
total = con.execute("SELECT COUNT(*) FROM features").fetchone()[0]
checks.append(("row_count = 810921",          total == 810_921,      total))

# Check 2 — sem duplicatas por supplier_sk
distinct_sk = con.execute("SELECT COUNT(DISTINCT supplier_sk) FROM features").fetchone()[0]
checks.append(("supplier_sk unique",           distinct_sk == total,  distinct_sk))

# Check 3 — sem nulls nas features binárias
null_binaries = con.execute("""
    SELECT COUNT(*) FROM features
    WHERE tem_sancao IS NULL
       OR tem_ceis   IS NULL
       OR tem_cnep   IS NULL
       OR sancao_ativa IS NULL
       OR tem_contrato_pncp IS NULL
       OR tem_contrato_compras IS NULL
""").fetchone()[0]
checks.append(("no nulls in binary features",  null_binaries == 0,    null_binaries))

# Check 4 — sem nulls nas features numéricas de contrato
null_numerics = con.execute("""
    SELECT COUNT(*) FROM features
    WHERE qtd_contratos IS NULL
       OR valor_total_contratos IS NULL
       OR valor_medio_contrato  IS NULL
""").fetchone()[0]
checks.append(("no nulls in numeric features", null_numerics == 0,    null_numerics))

# Check 5 — fornecedores sancionados batem com fato_sancoes
sancionados_features = con.execute("SELECT COUNT(*) FROM features WHERE tem_sancao = 1").fetchone()[0]
sancionados_fato     = con.execute("SELECT COUNT(DISTINCT supplier_sk) FROM fato_sancoes").fetchone()[0]
checks.append(("sancionados count match",      sancionados_features == sancionados_fato,
               f"{sancionados_features} vs {sancionados_fato}"))

# Check 6 — sem supplier_sk na features ausente na dim_fornecedor
orphans = con.execute("""
    SELECT COUNT(*) FROM features f
    WHERE NOT EXISTS (
        SELECT 1 FROM dim_fornecedor d
        WHERE d.supplier_sk = f.supplier_sk
    )
""").fetchone()[0]
checks.append(("no orphan supplier_sk",        orphans == 0,          orphans))

# Check 7 — cnpj_normalized não exposto (ADR-005)
cols = [r[0] for r in con.execute("DESCRIBE features").fetchall()]
checks.append(("cnpj_normalized not exposed",  "cnpj_normalized" not in cols, cols))

# ── resultado ─────────────────────────────────────────────────────────────────
print(f"{'Check':<35} {'Status':<8} {'Value'}")
print("-" * 70)
all_pass = True
for name, passed, value in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"{name:<35} {status:<8} {value}")

print("-" * 70)
print("ALL CHECKS PASSED ✅" if all_pass else "⚠️  SOME CHECKS FAILED — review above")

In [ ]:
## Step 8 — Export CSV for H2O Driverless AI

CSV_OUTPUT_PATH = str(OUTPUT_PATH).replace(".parquet", ".csv")

con.execute(f"""
    COPY (
        SELECT
            supplier_sk,
            porte,
            natureza_juridica_desc,
            uf,
            situacao_cadastral,
            qtd_contratos,
            qtd_orgaos_distintos,
            valor_total_contratos,
            valor_medio_contrato,
            valor_max_contrato,
            anos_ativo_contratos,
            tem_contrato_pncp,
            tem_contrato_compras,
            tem_sancao,
            qtd_sancoes,
            tem_ceis,
            tem_cnep,
            sancao_ativa,
            valor_total_multa
        FROM read_parquet('{OUTPUT_PATH}')
    )
    TO '{CSV_OUTPUT_PATH}'
    (FORMAT CSV, HEADER true)
""")

print(f"Written: {CSV_OUTPUT_PATH}")
print(f"File size: {Path(CSV_OUTPUT_PATH).stat().st_size / 1024 / 1024:.1f} MB")